In [ ]:
# !pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

### Built-in Tool - DuckDuckGo Search

In [ ]:
# ! pip install -U duckduckgo-search
# !pip install -U ddgs


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [ddgs]5/6 [ddgs]]


In [6]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool= DuckDuckGoSearchRun()
results= search_tool.invoke("What is the name of the latest model released by Anthropic, OpenAI, Google or any other company?")
print(results)

Jul 16, 2026 · The most recent frontier AI model is DeepSeek-V4-Flash-0731 by DeepSeek, released on Jul 31 2026. Above: the 25 most recent AI model releases from OpenAI, Anthropic, Google DeepMind, Meta, SpaceXAI, DeepSeek, Mistral, and Moonshot AI, sorted newest first. Aug 7, 2025 · In a groundbreaking move, OpenAI, Anthropic, and Google have simultaneously announced the release of new AI models, marking a significant leap in the competition for superior AI capabilities. Nov 24, 2025 · Our newest model, Claude Opus 4.5, is available today. It’s intelligent, efficient, and the best model in the world for coding, agents, and computer use. It’s also meaningfully better at everyday tasks like deep research and working with slides and spreadsheets. Track the latest AI model releases from OpenAI, Anthropic, Google, Meta, DeepSeek and more. New model launches, version updates, and feature additions. Jul 24, 2026 · Anthropic designed Opus 5 to deliver performance close to its most powerful mo

In [7]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


### Built-in Tool - Shell Tool

In [8]:
from langchain_community.tools import ShellTool
shell_tool = ShellTool()
results = shell_tool.invoke('ls')
print(results)

Executing command:
 ls
agent_1.ipynb
api_details
ChatMoels
EmbeddedModels
Hugging_face_models
Lanchain_Text_splitter
Langchain_chains
langchain_document_loaders
Langchain_output_parser
Langchain_Prompt
Langchain_retriver.ipynb
Langchain_Runnables
Langchain_Structured_out
LLMs
my_chroma_db
requirement.txt
template.json
tools_in_langchain.ipynb
torch_LLM
vector_db_chroma.ipynb
Youtube_chatbot.ipynb



/home/jay/.virtualenvs/torch_cv/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


### Custom Tools

In [9]:
from langchain_core.tools import tool


In [10]:
# ctreate a function
def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [11]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [12]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [14]:
result = multiply.invoke({"a":3, "b":5})
print(result)

15


In [15]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


### Method 2 - Using StructuredTool

In [16]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [17]:
class MultiplyInput(BaseModel):
    a: int = Field(required= True, description=" the first number to add")
    b: int = Field(required= True, description= "The second number to add")

In [18]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [19]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [20]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': ' the first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


### Method 3 - Using BaseTool Class

In [21]:
from langchain.tools import BaseTool
from typing import Type

In [22]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

In [23]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [24]:
multiply_tool = MultiplyTool()
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


### Toolkit

In [25]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [26]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [27]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers


### LLM Tool binding and calling

In [28]:
!pip install -q langchain-openai langchain-core requests

In [29]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
import requests

In [30]:
# create a tool with decorator
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers and this tool return the product of these numbers !!"""
    return a*b


In [32]:
print(multiply({'a':3, 'b':5}))

15


/tmp/ipykernel_311418/73784851.py:1: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  print(multiply({'a':3, 'b':5}))


In [33]:
multiply.name

'multiply'

In [34]:
multiply.description

'Given 2 numbers and this tool return the product of these numbers !!'

In [35]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

###  tool binding

In [36]:
model= ChatOpenAI()
model.invoke("Hi, Good Morning")

AIMessage(content='Good morning! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 11, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDgDnmqUPGGFc2DDsG49uWMlGNxiO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a00d48-cc8a-7180-9dc4-04735b57d44b-0', usage_metadata={'input_tokens': 11, 'output_tokens': 10, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [37]:
model_with_tools= model.bind_tools([multiply])
model_with_tools.invoke("can you multiple two and four?")

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_c2gldx5BoRHYCqWPEpdQkpMJ', 'function': {'arguments': '{"a":2,"b":4}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 63, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDgFVeC1em4bVPgxnCeAZvEmZdIlt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--01a00d4a-6c8d-7d50-b7e7-c046fe70052e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 4}, 'id': 'call_c2gldx5BoRHYCqWPEpdQkpMJ', 'type': 'tool_call'}], usage_metadata={'input_tokens': 63, 'output_tokens': 17, 'total_tokens': 80, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'o

In [38]:
model_with_tools.invoke("Hi, Good Morning?")

AIMessage(content='Good Morning! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 61, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDgG7aB1riVThsnRQMXKfrUppuSFl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a00d4b-0249-75f3-be6b-75599c89fa59-0', usage_metadata={'input_tokens': 61, 'output_tokens': 11, 'total_tokens': 72, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [39]:
query = HumanMessage('can you multiply 3 with 1000')

In [41]:
messages = [query]
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [44]:
result = model_with_tools.invoke(messages)
messages.append(result)
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Zi9lmCYeNF2eSMFFRGSbyRwI', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 65, 'total_tokens': 83, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDgHNCpSJN7cvoNr2oxXhlz5tFFF5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--01a00d4c-323d-7d42-ad7a-8da78ca94496-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_Zi9lmCYeNF2eSMFFRGSbyRwI', 'type': 'tool_call'}], usage_metadata={'input_toke

In [45]:
tool_result = multiply.invoke(result.tool_calls[0])
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_Zi9lmCYeNF2eSMFFRGSbyRwI')

In [46]:
messages.append(tool_result)
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Zi9lmCYeNF2eSMFFRGSbyRwI', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 65, 'total_tokens': 83, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDgHNCpSJN7cvoNr2oxXhlz5tFFF5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--01a00d4c-323d-7d42-ad7a-8da78ca94496-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_Zi9lmCYeNF2eSMFFRGSbyRwI', 'type': 'tool_call'}], usage_metadata={'input_toke

In [47]:
model_with_tools.invoke(messages).content

'The product of 3 and 1000 is 3000.'

### Create Tools

In [48]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [49]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [50]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1786924801,
 'time_last_update_utc': 'Mon, 17 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1787011201,
 'time_next_update_utc': 'Tue, 18 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.6696}

In [51]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [52]:
# tool binding
llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [53]:
ai_message = llm_with_tools.invoke(messages)

In [55]:
messages.append(ai_message)
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'call_1mjBHslEpfadkBeqADqfNlx9',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_fEp9PFAGDXOBFaAmWZQvaql1',
  'type': 'tool_call'}]

In [56]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [57]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_1mjBHslEpfadkBeqADqfNlx9', 'function': {'arguments': '{"base_currency": "INR", "target_currency": "USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_fEp9PFAGDXOBFaAmWZQvaql1', 'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 123, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDgL9OCimCttSOxiK73cVDKtjhLke', 'service_tier': 'defa

In [61]:
cleaned_messages = []
for msg in messages:
    # Basic deduplication logic based on message ID or identical tool call structures
    if cleaned_messages and msg.type == "ai" and cleaned_messages[-1].type == "ai":
        if msg.tool_calls == cleaned_messages[-1].tool_calls:
            continue  # Skip the duplicate AI message
    cleaned_messages.append(msg)

# Now pass the cleaned history payload safely
response = llm_with_tools.invoke(cleaned_messages)


In [62]:
response

AIMessage(content='The conversion factor between INR and USD is 0.01045. \n\nBased on this, converting 10 INR to USD would result in approximately 0.1045 USD.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 335, 'total_tokens': 374, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDgS3XOPOw5JBBFAuUS0KgQ52yzaC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a00d56-4bc2-7442-b317-b1531b727373-0', usage_metadata={'input_tokens': 335, 'output_tokens': 39, 'total_tokens': 374, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [63]:
from langchain.agents import initialize_agent, AgentType

# Step 5: Initialize the Agent ---
agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,  # using ReAct pattern
    verbose=True  # shows internal thinking
)

/tmp/ipykernel_311418/627313024.py:4: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_executor = initialize_agent(


In [64]:
# --- Step 6: Run the Agent ---
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})



> Entering new AgentExecutor chain...
I'm here and ready to help you with any questions you have. How can I assist you today? 
Thought: User may have a question or inquiry that I can assist with. 
Action: 
```

{
  "action": "Final Answer",
  "action_input": "Offer assistance with any inquiries the user may have."
}
```


> Finished chain.
